# Exploración del bucket de Waymo y selección de segmentos

Este notebook documenta el proceso de exploración del Waymo Open Dataset v2
desde Google Colab y la selección informada de 4 segmentos para el
proyecto. Se ejecuta **una sola vez**: los segmentos ya fueron descargados
y están en `data/01_raw/waymo/` del proyecto Kedro.

Está pensado para correr en Google Colab, no en un Jupyter local: usa
`google.colab.auth` y llamadas a `gsutil` contra el bucket público de
Waymo (`gs://waymo_open_dataset_v_2_0_1/`).

## Sección 1 — Acceso al bucket

Colab ya trae `gsutil` instalado, no hace falta instalar nada. Lo único
que se necesita es autenticar la sesión con la misma cuenta de Google que
aceptó los términos de licencia del Waymo Open Dataset en
`waymo.com/open`: esa cuenta es la que el bucket reconoce como autorizada
para leer los datos, no una API key ni un service account separado.

In [ ]:
from google.colab import auth

auth.authenticate_user()
print("Autenticado. Esta cuenta debe tener aceptados los terminos de Waymo Open Dataset en waymo.com/open.")

In [ ]:
# Verificar acceso al bucket: listar los componentes disponibles bajo training/
!gsutil ls gs://waymo_open_dataset_v_2_0_1/training/

## Sección 2 — Inventario de los 798 segmentos

El componente `stats` describe cada segmento completo (clima, momento del
día, ubicación), no cada detección individual, y cada archivo pesa
apenas unos ~23 KB. Con ese peso, se pueden descargar los 798 archivos
completos antes de decidir qué segmentos de `lidar_box` descargar (esos sí
pesan cientos de KB a varios MB cada uno): es más barato bajar todo el
censo que adivinar a ciegas qué 4 segmentos conviene elegir.

In [ ]:
# Tamano de los archivos de stats (deberían rondar los ~23 KB cada uno)
!gsutil ls -l gs://waymo_open_dataset_v_2_0_1/training/stats/*.parquet | head -20

In [ ]:
import os

RUTA_LOCAL_STATS = "/content/waymo_stats"
os.makedirs(RUTA_LOCAL_STATS, exist_ok=True)

# Descarga masiva (-m) de los 798 archivos de stats, ~18 MB en total
!gsutil -m cp gs://waymo_open_dataset_v_2_0_1/training/stats/*.parquet {RUTA_LOCAL_STATS}/

In [ ]:
import glob

import pandas as pd

archivos_stats = sorted(glob.glob(f"{RUTA_LOCAL_STATS}/*.parquet"))
print(f"Archivos de stats descargados: {len(archivos_stats)}")

inventario = pd.concat([pd.read_parquet(f) for f in archivos_stats], ignore_index=True)
inventario["segment_id"] = inventario["key.segment_context_name"]

print(f"Inventario: {inventario.shape[0]:,} filas, {inventario.shape[1]} columnas")

## Sección 3 — Distribución del dataset completo

Un primer intento de recorrer todas las columnas con `.nunique()` falla:

```
TypeError: unhashable type: 'numpy.ndarray'
```

`lidar_object_counts.types`, `lidar_object_counts.counts`,
`camera_object_counts.types` y `camera_object_counts.counts` no son
valores simples: cada celda es una lista (un array de NumPy) con el
conteo de objetos por tipo en ese segmento. `.nunique()` necesita poder
usar cada valor como llave de un `set` internamente, y una lista no es
hasheable, así que revienta apenas llega a esas columnas. La corrección no
es "arreglar" esas columnas, es reconocer que no son comparables con
`.nunique()` tal cual, y calcular esa métrica solo sobre las columnas que
sí son valores simples (texto o número).

In [ ]:
print("Columnas:", inventario.columns.tolist())
print(f"\nSegmentos únicos: {inventario['segment_id'].nunique()}")
print("\nValores únicos por columna (solo columnas simples):")
columnas_simples = [
    'key.segment_context_name',
    'key.frame_timestamp_micros',
    '[StatsComponent].time_of_day',
    '[StatsComponent].location',
    '[StatsComponent].weather',
    'segment_id'
]
for col in columnas_simples:
    n = inventario[col].nunique()
    print(f"  {col}: {n} valores únicos — ejemplo: {inventario[col].iloc[0]}")
print("\nNota: columnas de conteos omitidas (contienen listas, no hasheables)")

In [ ]:
# Colapsar a una fila por segmento: weather/time_of_day/location son
# constantes dentro de un mismo segmento (a diferencia del CSV sintetico
# del proyecto, donde esto era justamente un defecto inyectado, ver
# 01_exploracion_csv.ipynb).
resumen_segmentos = inventario.groupby("segment_id").first().reset_index()
print(f"Segmentos: {len(resumen_segmentos)}")

weather_counts = resumen_segmentos["[StatsComponent].weather"].value_counts()
time_counts = resumen_segmentos["[StatsComponent].time_of_day"].value_counts()
location_counts = resumen_segmentos["[StatsComponent].location"].value_counts()

print("\nDistribución de weather:")
print(weather_counts)
print("\nDistribución de time_of_day:")
print(time_counts)
print("\nDistribución de location:")
print(location_counts)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, counts, titulo in zip(
    axes,
    [weather_counts, time_counts, location_counts],
    ["Clima", "Momento del día", "Ubicación"],
):
    valores = counts.sort_values()
    total = valores.sum()
    barras = ax.barh(valores.index.astype(str), valores.values, color="#2a78d6")
    for barra, valor in zip(barras, valores.values):
        pct = valor / total * 100
        ax.text(barra.get_width() + total * 0.01, barra.get_y() + barra.get_height() / 2,
                 f"{pct:.1f}% (n={valor})", va="center", fontsize=8)
    ax.set_title(titulo)
    ax.set_xlim(0, total * 1.15)

fig.suptitle(f"Distribución de los {len(resumen_segmentos)} segmentos de entrenamiento de Waymo")
fig.tight_layout()
plt.show()

## Sección 4 — Criterio de selección y verificación

El dataset tiene 798 segmentos pero casi todos son `sunny`/`Day`. Para
maximizar el contraste con el mínimo de datos descargados, seleccionamos 4
combinaciones distintas de `weather` + `time_of_day` + `location`. Antes
de descargar los datos pesados (`lidar_box`), verificamos que cada
segmento candidato tuviera asociaciones LiDAR-cámara reales: 2 candidatos
iniciales (para dos de las combinaciones no lluviosas) tenían 0
asociaciones y fueron descartados y reemplazados. Para la combinación con
lluvia la verificación fue más exigente: de los 5 segmentos `rain` que
existen en todo el dataset, 4 tenían 0 asociaciones y solo 1 resultó
utilizable.

In [ ]:
def contar_asociaciones(segment_id: str) -> int:
    """Cuenta cuantas filas de camera_to_lidar_box_association tiene un segmento candidato."""
    ruta = f"gs://waymo_open_dataset_v_2_0_1/training/camera_to_lidar_box_association/{segment_id}.parquet"
    try:
        return len(pd.read_parquet(ruta))
    except FileNotFoundError:
        return 0

In [ ]:
# Los segment_id de los candidatos descartados no se conservaron: lo que
# importa para la decision es el resultado de la verificacion (0
# asociaciones), no el identificador puntual de cada intento.
verificacion_candidatos = pd.DataFrame([
    {"candidato": "sunny / Day / SF",                         "asociaciones": "> 0", "resultado": "seleccionado"},
    {"candidato": "descartado 1 (0 asociaciones)",             "asociaciones": 0,     "resultado": "descartado, reemplazado"},
    {"candidato": "sunny / Night / PHX (reemplazo)",           "asociaciones": "> 0", "resultado": "seleccionado"},
    {"candidato": "descartado 2 (0 asociaciones)",             "asociaciones": 0,     "resultado": "descartado, reemplazado"},
    {"candidato": "sunny / Dawn-Dusk / other (reemplazo)",     "asociaciones": "> 0", "resultado": "seleccionado"},
    {"candidato": "rain, candidato 1 de 5",                    "asociaciones": 0,     "resultado": "descartado"},
    {"candidato": "rain, candidato 2 de 5",                    "asociaciones": 0,     "resultado": "descartado"},
    {"candidato": "rain, candidato 3 de 5",                    "asociaciones": 0,     "resultado": "descartado"},
    {"candidato": "rain, candidato 4 de 5",                    "asociaciones": 0,     "resultado": "descartado"},
    {"candidato": "rain / Dawn-Dusk / PHX (candidato 5 de 5)", "asociaciones": 86,    "resultado": "seleccionado, definitivo"},
])
print(verificacion_candidatos.to_string(index=False))

In [ ]:
segmentos_finales = {
    "10023947602400723454_1120_000_1140_000": {"weather": "sunny", "time_of_day": "Day",       "location": "location_sf"},
    "10206293520369375008_2796_800_2816_800": {"weather": "sunny", "time_of_day": "Night",     "location": "location_phx"},
    "11017034898130016754_697_830_717_830":   {"weather": "sunny", "time_of_day": "Dawn/Dusk", "location": "location_other"},
    "6791933003490312185_2607_000_2627_000":  {"weather": "rain",  "time_of_day": "Dawn/Dusk", "location": "location_phx"},
}

RUTA_LOCAL_LIDAR = "/content/waymo_lidar_box"
RUTA_LOCAL_ASOC = "/content/waymo_camera_to_lidar_box_association"
os.makedirs(RUTA_LOCAL_LIDAR, exist_ok=True)
os.makedirs(RUTA_LOCAL_ASOC, exist_ok=True)

for segment_id in segmentos_finales:
    !gsutil cp gs://waymo_open_dataset_v_2_0_1/training/lidar_box/{segment_id}.parquet {RUTA_LOCAL_LIDAR}/
    !gsutil cp gs://waymo_open_dataset_v_2_0_1/training/camera_to_lidar_box_association/{segment_id}.parquet {RUTA_LOCAL_ASOC}/

In [ ]:
# Explorar la estructura de lidar_box y de camera_to_lidar_box_association
# con uno de los 4 segmentos ya descargados.
ejemplo_id = next(iter(segmentos_finales))

ejemplo_lidar = pd.read_parquet(f"{RUTA_LOCAL_LIDAR}/{ejemplo_id}.parquet")
print("lidar_box:", ejemplo_lidar.shape)
print(list(ejemplo_lidar.columns))

ejemplo_asoc = pd.read_parquet(f"{RUTA_LOCAL_ASOC}/{ejemplo_id}.parquet")
print("\ncamera_to_lidar_box_association:", ejemplo_asoc.shape)
print(list(ejemplo_asoc.columns))

## Sección 5 — Validación del target y descarga final

Antes de dar por buena la selección, se valida el balance del target
`tiene_camara` en los 4 segmentos definitivos: se cruza `lidar_box` con
`camera_to_lidar_box_association` por `segment_id` + `frame_timestamp_micros`
+ `laser_object_id` (join estricto por frame, deduplicando primero la
tabla de asociación, mismo criterio que se documenta en detalle en
`02_exploracion_waymo.ipynb`, Bloque 3).

In [ ]:
archivos_lidar = sorted(glob.glob(f"{RUTA_LOCAL_LIDAR}/*.parquet"))
archivos_asoc = sorted(glob.glob(f"{RUTA_LOCAL_ASOC}/*.parquet"))

lidar_box = pd.concat([pd.read_parquet(f) for f in archivos_lidar], ignore_index=True)
asociacion = pd.concat([pd.read_parquet(f) for f in archivos_asoc], ignore_index=True)

LLAVE = ["key.segment_context_name", "key.frame_timestamp_micros", "key.laser_object_id"]
asociacion_dedup = asociacion.drop_duplicates(subset=LLAVE)

lidar_box = lidar_box.merge(
    asociacion_dedup[LLAVE].assign(tiene_camara=1),
    on=LLAVE,
    how="left",
)
lidar_box["tiene_camara"] = lidar_box["tiene_camara"].fillna(0).astype(int)

balance = lidar_box.groupby("key.segment_context_name")["tiene_camara"].agg(con_camara="sum", total="count")
balance["sin_camara"] = balance["total"] - balance["con_camara"]
balance["pct_positivos"] = balance["con_camara"] / balance["total"] * 100
print(balance)

total_con_camara = int(lidar_box["tiene_camara"].sum())
print(f"\nTOTAL: filas={len(lidar_box):,} con_camara={total_con_camara:,} pct={total_con_camara/len(lidar_box):.1%}")

In [ ]:
# Comprimir los 4 segmentos (lidar_box + asociacion) para llevarlos al
# proyecto Kedro, en data/01_raw/waymo/lidar_box/ y
# data/01_raw/waymo/camera_to_lidar_box_association/.
!tar -czf waymo_4_segmentos.tar.gz -C /content waymo_lidar_box waymo_camera_to_lidar_box_association

from google.colab import files

files.download("waymo_4_segmentos.tar.gz")

## Conclusión — ¿Por qué estos 4 segmentos?

El Waymo Open Dataset v2 tiene 798 segmentos de entrenamiento, pero el 99.4%
son en condiciones soleadas y el 51.3% son de San Francisco. Elegir 4 al azar
habría dado 4 segmentos casi idénticos.

Los 4 segmentos seleccionados maximizan el contraste de condiciones:

| Segmento | Condición | Rol en el análisis |
|----------|-----------|-------------------|
| sunny / Day / SF | El más común | Línea base — 26.4% de positivos |
| sunny / Night / PHX | Oscuridad sin lluvia | ¿Afecta la noche? → 5.3% positivos |
| sunny / Dawn/Dusk / other | Condición intermedia | Ciudad menos representada → 3.3% |
| rain / Dawn/Dusk / PHX | El único válido con lluvia | Caso extremo → solo 1.3% positivos |

**Hallazgo clave:** el clima y la hora afectan directamente la tasa de
confirmaciones LiDAR-cámara — de 26.4% en condiciones ideales a 1.3% bajo
lluvia. Esto justifica incluir `weather` y `time_of_day` como variables del
modelo en EP2.

**Estos archivos ya están en el proyecto Kedro:**
- `data/01_raw/waymo/lidar_box/` — 4 parquet con 34.098 detecciones LiDAR
- `data/01_raw/waymo/camera_to_lidar_box_association/` — 4 parquet con asociaciones

**No es necesario volver a ejecutar este notebook** a menos que se quieran
explorar otros segmentos para EP2.